# Retrieval Failure Analysis

This notebook investigates cases where the correct PubMedQA source document was not ranked first by the retriever.

The baseline retrieval results were Recall@1 = 0.970, Recall@3 = 0.988, Recall@5 = 0.990, and MRR = 0.979. Rather than changing the system immediately, we inspect the failures to understand what the metrics are telling us.

## 1. What counts as a failure?

For each of the 500 source documents, the associated PubMedQA question is used as a query. A rank-1 failure means the first retrieved chunk comes from a different `pubid`.

This is a source-level retrieval test. It does **not** yet prove that the retrieved text contains every piece of information required to answer the question.

## 2. Run the failure analysis

The analysis writes all rank-1 failures to `data/retrieval_failures.json` so they can be reviewed without rerunning the retrieval process.

In [1]:
import json
from pathlib import Path

failures_path = Path("../data/retrieval_failures.json")

with failures_path.open() as f:
    failures = json.load(f)

print(f"Rank-1 failures: {len(failures)}")

Rank-1 failures: 15


## 3. Inspect the failures

Review the question and the retrieved candidates. Pay particular attention to:

- whether the correct source appears at rank 2–5;
- whether the top result is semantically similar but from a different study;
- whether the question contains terminology that the embedding model handles poorly;
- whether the relevant information appears to be distributed across multiple chunks.

In [2]:
for failure in failures:
    print("=" * 80)
    print("PubID:", failure["pubid"])
    print("Correct source rank:", failure["correct_source_rank"])
    print("Question:", failure["question"])
    print()
    for result in failure["retrieved"]:
        print(
            f"Rank {result['rank']} | "
            f"score={result['score']} | "
            f"pubid={result['pubid']} | "
            f"chunk={result['chunk_index']}"
        )
        print(result["text"][:700])
        print()

PubID: 23831910
Correct source rank: 2
Question: Double balloon enteroscopy: is it efficacious and safe in a community setting?

Rank 1 | score=0.6065 | pubid=25251991 | chunk=0
There are few data concerning emergency double-balloon enteroscopy (DBE) and its usefulness in the management of severe acute obscure gastrointestinal bleeding (OGIB). The aim of this retrospective study was to evaluate emergency DBE and capsule endoscopy (CE) in patients with overt OGIB, analyzing the feasibility of this combined approach. Emergency DBE in patients with overt OGIB was defined as performance within 24 h of symptom onset. We reported 27 patients (16 men, mean age: 64.6 ± 17.9 years) with overt severe bleeding who underwent 29 emergency DBE (22 anterograde, 7 retrograde). Of 27 patients, 16 (59.3%) underwent CE with real time (RT) viewing.

Rank 2 | score=0.4118 | pubid=23831910 | chunk=0
From March 2007 to January 2011, 88 DBE procedures were performed on 66 patients. Indications included evalua

## 4. Engineering conclusion

The purpose of this analysis is not to optimise the system against a handful of examples. It is to establish whether the baseline failures reveal a systematic weakness.

If most failures still retrieve the correct source within the top 3–5 results, the baseline is already strong and the next improvement should focus on whether the retrieved context is sufficient for answer generation.

If failures show a recurring pattern—such as poor handling of biomedical terminology, overly broad chunks, or relevant information split across chunks—we can use that evidence to choose the next experiment.

## 5. Next stage

After reviewing the failures, the project can move to answer generation and end-to-end RAG evaluation. Retrieval quality and answer quality should remain separate measurements.

## Failure Analysis Findings

The baseline evaluation identified 15 rank-1 failures. These should not all be treated as equivalent retrieval failures.

Of the 15 cases, 10 still retrieved the correct PubMed source within the top five results: 7 at rank 2, 2 at rank 3, and 1 at rank 4. Only 5 cases were genuine top-5 misses. This is consistent with the overall Recall@5 baseline of 0.990.

The near-misses are mostly cases where another paper on a closely related biomedical topic receives a higher similarity score. Examples include anticoagulation after intracerebral haemorrhage versus other anticoagulation studies, prostate-cancer questions versus other prostate-cancer studies, and esophageal mucosal fragility versus other esophageal conditions. This indicates semantic competition between related documents rather than an obvious indexing failure.

Several failures also involve very short or generic questions, such as questions equivalent to “Is it Crohn's disease?” or “Is it better to be big?”. These provide limited retrieval signal, making it difficult for a general-purpose embedding model to distinguish the intended paper from other medically related papers.

The five genuine top-5 misses are more important for the next iteration. They show cases where the retriever remains in the general biomedical/topic neighbourhood but does not surface the intended source among the first five results.

### Engineering conclusion

The current evidence does not justify changing the context-preserving chunking strategy. The correct source is frequently retrieved, including in several of the failure cases, and the failures are more strongly associated with semantic similarity between competing biomedical documents and limited query signal.

The next controlled experiment should therefore test the embedding model while keeping the corpus, chunks, queries, vector-store configuration, and evaluation metrics unchanged. A biomedical-domain embedding model can then be compared directly against `all-MiniLM-L6-v2` using Recall@1, Recall@3, Recall@5, and MRR.

This creates a clean engineering experiment: change one major retrieval variable and measure whether biomedical-domain embeddings reduce the observed near-misses and genuine misses.


## Evaluation Limitations Identified

The current relevance definition is source-level: a retrieved chunk is considered relevant when its `pubid` matches the query's source `pubid`. This is appropriate as an initial retrieval benchmark, but it does not establish that the retrieved chunk contains the evidence needed to answer the question. Multiple chunks from the same paper are therefore treated as equally relevant.

A later evaluation stage should introduce a more granular notion of relevance, using the underlying PubMedQA context/evidence so that retrieval quality can be assessed at the answer-bearing context level.

The current benchmark also uses questions whose source documents are present in the indexed corpus. This is useful for measuring whether the system can retrieve the originating document, but it should be kept distinct from a held-out generalization benchmark when interpreting future improvements.

These limitations do not invalidate the baseline; they define what the current metrics do and do not demonstrate.
